# Colab PP1: hybrid heads + Laya in one T4 session

**J26-SE-325, Component 1.** A reduced-size run of Part A (`colab_hybrid_heads.ipynb`) and
Part C (`colab_laya_decision.ipynb`) back to back, sized to finish in one session (about
2-3 hours on a T4). Same code, same splits, same leak guards; only the universe and
iteration counts are smaller. Results are **preliminary** and should be labelled so.

Runtime > Change runtime type > **T4 GPU**, then Run all. Outputs go to Drive, so a
disconnect loses nothing: re-run all and finished stages are skipped.

| Knob | Full run | This run |
|---|---|---|
| CSE securities | all with >= 200 sessions | first 60 |
| Walk-forward points / symbol | 120 | 60 |
| Head epochs | 30 | 20 |
| Laya scenarios | 20,000 | 3,000 |
| Laya epochs | 3 | 2 |

**Before running:** put `fundamentals_backfill.parquet` on Drive at
`MyDrive/j26_se_325_backfill/` (copy `artifacts/backfill/fundamentals_backfill.parquet` from the
laptop, or run `colab_fundamentals_backfill.ipynb`). Both trainers below read it. Without it the
fundamentals inputs are zero-filled and Laya's fundamental questions get no rows; the next
cell says which case you are in.


In [ ]:
# === BOOTSTRAP (same contract as colab_finetune.ipynb) ================================
# Clone or update the branch, chdir into the component, install PIP_PKGS. Safe to re-run.
REPO_URL = "https://github.com/SE-Y4S1/J26-SE-325.git"
BRANCH = "Nivakaran"  # the component is NOT on main
SUBDIR = "backend/Portfolio-Optimization"
PIP_PKGS = "timesfm[torch]==2.0.2 laya==0.4.1 pandas-ta-openbb yfinance pypdf pyyaml requests pydantic pydantic-settings scikit-fuzzy"

import importlib, os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
OWN_PACKAGES = {"agent", "configs", "data", "evaluation", "experiments", "features",
                "forecasting", "fundamentals", "news", "optimization", "decision", "service"}


def find_root(start):
    for base in (start, *start.parents):
        if (base / "configs/universe.yaml").exists() and (base / "fundamentals").exists():
            return base
    return None


# In Colab always fetch: a re-run starts INSIDE the checkout, so find_root
# succeeded and pushed fixes never reached a live runtime.
root = None if IN_COLAB else find_root(Path.cwd())
if root is None:
    if not IN_COLAB:
        raise SystemExit("cd into backend/Portfolio-Optimization first.")
    checkout = Path("/content/J26-SE-325")
    if (checkout / SUBDIR).exists():
        if subprocess.run(["git", "-C", str(checkout), "fetch", "--depth", "1", "origin", BRANCH]).returncode == 0:
            subprocess.run(["git", "-C", str(checkout), "reset", "--hard", "FETCH_HEAD"])
    else:
        if subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(checkout)]).returncode:
            from getpass import getpass
            token = getpass("Clone failed. GitHub token (repo scope), blank to abort: ").strip()
            if not token:
                raise SystemExit("Cannot reach " + REPO_URL)
            subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                            REPO_URL.replace("https://", "https://" + token + "@"), str(checkout)], check=True)
    root = checkout / SUBDIR

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
for _name in [n for n in sys.modules if n.split(".")[0] in OWN_PACKAGES]:
    del sys.modules[_name]
importlib.invalidate_caches()
if IN_COLAB and PIP_PKGS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + PIP_PKGS.split(), check=True)
print("OK root=" + str(root))

In [ ]:
import torch

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"

from forecasting.env_report import write_env_report

report = write_env_report()
assert report["optional_forecasters"]["timesfm"]["available"], report["optional_forecasters"]["timesfm"]
import laya

print("laya", laya.__version__)

In [ ]:
from pathlib import Path

ROOT = Path("artifacts/pp1")
BACKFILL = Path("artifacts/backfill/fundamentals_backfill.parquet")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/j26_pp1")
    BACKFILL = Path("/content/drive/MyDrive/j26_se_325_backfill/fundamentals_backfill.parquet")
HYBRID, LAYA = ROOT / "hybrid", ROOT / "laya"
HYBRID.mkdir(parents=True, exist_ok=True)
LAYA.mkdir(parents=True, exist_ok=True)
print("out:", ROOT)
print("fundamentals backfill:",
      BACKFILL if BACKFILL.exists() else
      "NOT FOUND -- fundamentals are zero-filled and Laya's fundamental questions get no rows")


## 1. Hybrid heads (h = 5 and 21)

The holdout step scores zero-shot TimesFM and the hybrid on the same rows: both RQ1 rows.

In [ ]:
from experiments.train_hybrid_heads import main as train_hybrid

train_hybrid(["--out", str(HYBRID), "--cse-limit", "60", "--horizons", "5", "21",
              "--walk-forward-points", "60", "--epochs", "20"]
             + (["--backfill", str(BACKFILL)] if BACKFILL.exists() else []))


In [ ]:
import json

import pandas as pd

rows = []
for head in sorted(HYBRID.glob("h*/meta.json")):
    meta = json.loads(head.read_text())
    m = meta["metrics"]
    for key in m:
        if key.startswith("pinball_base_"):
            cls = key.removeprefix("pinball_base_")
            rows.append({"horizon": meta["horizon"], "asset_class": cls, "timesfm_zero_shot": m[key],
                         "hybrid": m.get("pinball_hybrid_" + cls), "rows": m.get("holdout_rows_" + cls)})
rq1 = pd.DataFrame(rows)
rq1["improvement_%"] = 100 * (1 - rq1["hybrid"] / rq1["timesfm_zero_shot"])
rq1.to_csv(ROOT / "rq1_hybrid_holdout.csv", index=False)
rq1.round(5)

## 2. Laya decision model (RQ5)

Reuses the bars Part 1 just fetched.

In [ ]:
from experiments.train_laya import main as train_laya

train_laya(["--out", str(LAYA), "--bars", str(HYBRID / "bars.parquet"),
            "--scenarios", "3000", "--sample-every", "20", "--epochs", "2"]
           + (["--backfill", str(BACKFILL)] if BACKFILL.exists() else []))


In [ ]:
rq5 = pd.read_csv(LAYA / "evaluation.csv")
rq5.to_csv(ROOT / "rq5_laya.csv", index=False)
rq5.pivot(index="question", columns="model", values=["accuracy", "brier"]).round(3)

## 3. Bring it home

Download from Drive `j26_pp1/hybrid/hybrid_heads.zip` and `j26_pp1/laya/laya_head.zip`
(a few MB together), plus the two CSVs, then on the laptop:

```bash
python -m forecasting.serving --install hybrid_heads.zip
python -m decision.checkpoint --install laya_head.zip
```

In [ ]:
for p in [HYBRID / "hybrid_heads.zip", LAYA / "laya_head.zip", ROOT / "rq1_hybrid_holdout.csv", ROOT / "rq5_laya.csv"]:
    print(("OK      " if p.exists() else "MISSING ") + str(p))